# Notebook 10 — Análisis de series temporales y forecasting

## Objetivo

Aplicar el conjunto clásico de técnicas de análisis de series temporales sobre la facturación retail semanal de Selmark del cuatrienio 2022-2025. El notebook integra cuatro bloques complementarios: descomposición estacional STL, análisis de autocorrelaciones (ACF y PACF), tests de estacionariedad y ajuste de un modelo SARIMA con forecasting de 52 semanas para el año 2026.

Este análisis, sugerido por el tutor académico, complementa el análisis descriptivo desarrollado en los notebooks 09a y 09b, aportando rigor estadístico al estudio temporal del negocio y constituye una pieza diferencial del Trabajo de Fin de Grado por su carácter cuantitativo y predictivo.

## Estructura

1. Configuración y construcción de la serie temporal semanal
2. Descomposición STL
3. Análisis de autocorrelaciones (ACF y PACF)
4. Tests de estacionariedad
5. Modelo SARIMA y forecasting de 52 semanas
6. Síntesis y aportaciones al TFG

## Decisiones metodológicas

Se trabaja sobre la facturación retail total agregada por semana ISO, excluyendo las tres cuentas técnicas identificadas en el notebook 09a (REGO, HERREROS y EL CORTE INGLES) por su impacto distorsionador sobre los agregados. La granularidad semanal proporciona aproximadamente 209 observaciones, suficientes para un modelo SARIMA con periodo estacional s=52.

## 1. Configuración y construcción de la serie temporal

In [90]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Bibliotecas de series temporales
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import jarque_bera
from sklearn.metrics import mean_squared_error
import itertools

try:
    from pmdarima import auto_arima
    PMDARIMA_DISPONIBLE = True
except ImportError:
    PMDARIMA_DISPONIBLE = False

# Rutas
RUTA_PROYECTO = Path.home() / "OneDrive" / "Documentos" / "TFG_Selmark"
RUTA_DUCKDB   = RUTA_PROYECTO / "duckdb" / "selmark.duckdb"
RUTA_FIGURAS  = RUTA_PROYECTO / "output" / "figuras" / "10"
RUTA_FIGURAS.mkdir(parents=True, exist_ok=True)

# Paleta corporativa coherente
COLOR_NACIONAL       = "#C8447F"
COLOR_INTERNACIONAL  = "#3E5C76"
COLOR_DESTACAR       = "#D4A017"
COLOR_POSITIVO       = "#5C9E76"
COLOR_NEGATIVO       = "#B85450"
COLOR_NEUTRO         = "#7D7D7D"
PALETA = ["#C8447F", "#3E5C76", "#D4A017", "#5C9E76", "#B85450", "#7D7D7D"]

TEMPLATE = go.layout.Template(layout=dict(
    font=dict(family="Arial, Helvetica, sans-serif", size=12, color="#1A1A1A"),
    title=dict(font=dict(size=16, color="#1A1A1A"), x=0.02, xanchor="left"),
    plot_bgcolor="white", paper_bgcolor="white", colorway=PALETA,
    xaxis=dict(showgrid=True, gridcolor="#EAEAEA", zeroline=False, linecolor="#333333", linewidth=0.8),
    yaxis=dict(showgrid=True, gridcolor="#EAEAEA", zeroline=False, linecolor="#333333", linewidth=0.8),
    margin=dict(l=60, r=40, t=70, b=60),
))
pio.templates["selmark"] = TEMPLATE
pio.templates.default = "selmark"

con = duckdb.connect(str(RUTA_DUCKDB), read_only=True)

def guardar_y_mostrar(fig, nombre, w=950, h=500):
    ruta = RUTA_FIGURAS / f"{nombre}.png"
    fig.write_image(str(ruta), width=w, height=h, scale=2)
    fig.show()
    print(f"   Figura guardada: {ruta.name}")

def fmt_es(n, decimales=0):
    if pd.isna(n):
        return "—"
    if decimales == 0:
        return f"{int(n):,}".replace(",", ".")
    return f"{n:,.{decimales}f}".replace(",", "X").replace(".", ",").replace("X", ".")

print(f"Conexión: {RUTA_DUCKDB.name}")
print(f"Figuras se guardarán en: {RUTA_FIGURAS}")
print(f"pmdarima disponible: {PMDARIMA_DISPONIBLE}")

Conexión: selmark.duckdb
Figuras se guardarán en: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\output\figuras\10
pmdarima disponible: True


In [91]:
# 1.1 Construcción de la serie temporal semanal de facturación retail
# Agregación por (año, semana ISO) excluyendo cuentas técnicas
serie_raw = con.execute("""
    SELECT
        t.anio                                         AS anio,
        t.semana_iso                                   AS semana,
        MIN(vm.fecha_venta)                            AS fecha_inicio,
        MAX(vm.fecha_venta)                            AS fecha_fin,
        SUM(vm.importe_total_con_descuento)            AS facturacion_neta,
        COUNT(*)                                       AS num_operaciones,
        COUNT(DISTINCT vm.id_cliente)                  AS clientes_activos
    FROM silver.ventas_minoristas vm
    INNER JOIN silver.tiempo t ON vm.fecha_venta = t.fecha
    WHERE vm.es_devolucion = FALSE
      AND vm.id_cliente NOT IN ('5728', '30942', '30957')
      AND t.anio BETWEEN 2022 AND 2025
    GROUP BY t.anio, t.semana_iso
    ORDER BY t.anio, t.semana_iso
""").fetchdf()

print(f"Semanas obtenidas: {len(serie_raw)}")
print(f"Periodo: {serie_raw['fecha_inicio'].min().date()} a {serie_raw['fecha_fin'].max().date()}")
print(f"Facturación acumulada: {fmt_es(serie_raw['facturacion_neta'].sum()/1e6, 2)} M €")

Semanas obtenidas: 208
Periodo: 2022-01-01 a 2025-12-31
Facturación acumulada: 67,10 M €


In [92]:
# 1.2 Construcción del DataFrame indexado por fecha (lunes de la semana)
# Para SARIMA necesitamos un índice temporal regular sin saltos

# Tomamos el lunes de cada semana ISO como timestamp representativo
serie_raw["fecha_lunes"] = serie_raw.apply(
    lambda r: pd.Timestamp.fromisocalendar(int(r["anio"]), int(r["semana"]), 1),
    axis=1
)

# Construir un índice semanal completo y reindexar (para detectar y rellenar huecos)
idx_completo = pd.date_range(
    start=serie_raw["fecha_lunes"].min(),
    end=serie_raw["fecha_lunes"].max(),
    freq="W-MON",
)

serie = (serie_raw.set_index("fecha_lunes")["facturacion_neta"]
                  .reindex(idx_completo)
                  .rename("facturacion_neta"))
serie.index.name = "fecha"

# Diagnóstico
n_nulos = int(serie.isna().sum())
print(f"Observaciones en serie regular: {len(serie)}")
print(f"Semanas sin datos (huecos):     {n_nulos}")

if n_nulos > 0:
    print(f"\nHuecos detectados. Se imputan por interpolación lineal para STL.")
    serie = serie.interpolate(method="linear")
else:
    print(f"   La serie es completa, sin huecos.")

print(f"\nEstadísticas descriptivas (€):")
print(serie.describe().apply(lambda v: fmt_es(v, 2)).to_string())

Observaciones en serie regular: 208
Semanas sin datos (huecos):     0
   La serie es completa, sin huecos.

Estadísticas descriptivas (€):
count          208,00
mean       322.613,27
std        243.401,09
min         13.068,22
25%        137.546,55
50%        244.283,64
75%        472.474,07
max      1.523.508,12


In [93]:
# 1.3 Inspección visual inicial de la serie completa
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=serie.index, y=serie.values / 1e3,
    mode="lines", line=dict(color=COLOR_NACIONAL, width=1.6),
    name="Facturación semanal",
    hovertemplate="<b>Semana de %{x|%d %b %Y}</b><br>Facturación: %{y:,.1f} K €<extra></extra>"
))

# Líneas verticales en cada cambio de año (usando strings ISO para evitar error de kaleido)
for anio in range(2023, 2026):
    fecha_str = f"{anio}-01-01"
    fig.add_vline(x=fecha_str, line_dash="dot", line_color="#CCCCCC", line_width=1)
    fig.add_annotation(x=fecha_str,
                        y=1.02, yref="paper", text=str(anio),
                        showarrow=False, font=dict(size=10, color="#888"))

fig.update_layout(
    title=dict(text="<b>Serie temporal semanal de la facturación retail 2022–2025</b><br><sup>Datos agregados por semana ISO · sin cuentas técnicas</sup>"),
    xaxis_title="Fecha (lunes de cada semana ISO)",
    yaxis_title="Facturación retail neta (miles de €)",
    height=500,
)
guardar_y_mostrar(fig, "01_serie_completa")

   Figura guardada: 01_serie_completa.png


### Conclusiones de la inspección inicial

La serie temporal semanal de facturación retail presenta tres rasgos característicos que se observan visualmente y que se analizarán formalmente en los siguientes bloques: una estacionalidad anual marcada con picos sistemáticos en los meses de campaña (julio y septiembre principalmente), una alta variabilidad semana a semana derivada de los eventos comerciales puntuales (San Valentín, Black Friday, rebajas) y una posible tendencia ascendente leve que requerirá confirmación mediante descomposición y tests estadísticos.

#### Hallazgo crítico: tendencia descendente del retail

La inspección de los totales anuales revela un hallazgo de alta relevancia comercial que merece ser documentado explícitamente: la facturación retail anual de Selmark (sin cuentas técnicas) presenta una tendencia descendente sostenida durante el cuatrienio analizado.

| Año | Facturación retail | Variación interanual |
|---|---|---|
| 2022 | 17,42 M € | — (base) |
| 2023 | 16,98 M € | −2,5 % |
| 2024 | 16,48 M € | −3,0 % |
| 2025 | 16,22 M € | −1,6 % |
| **Total 4 años** | | **−6,9 %** |

La descomposición STL que se aplica a continuación atribuirá a esta tendencia un porcentaje pequeño de la varianza total porque su magnitud absoluta es modesta frente a la fuerte estacionalidad. Sin embargo, **se trata de una caída sostenida** que debe ser comunicada a Selmark como hallazgo estratégico y que será especialmente relevante en el Capítulo 8 (recomendaciones) y en el análisis de churn del Capítulo 7. La estabilidad estadística de la serie (estacionariedad) no debe confundirse con la salud del negocio: una serie puede ser estadísticamente estacionaria en torno a una media que está disminuyendo lentamente.

## 2. Descomposición STL

La descomposición STL (Seasonal-Trend decomposition using Loess) separa la serie temporal en tres componentes aditivos: la tendencia, la estacionalidad y los residuos. A diferencia de la descomposición clásica, STL es robusta a outliers y permite definir la estacionalidad de forma flexible, lo que la hace especialmente adecuada para series con eventos puntuales muy marcados como la de Selmark. Se aplica con un periodo estacional s = 52 (anual sobre frecuencia semanal).

In [94]:
# 2.1 Aplicación de la descomposición STL
stl = STL(serie, period=52, robust=True)
result = stl.fit()

trend     = result.trend
seasonal  = result.seasonal
residuals = result.resid

print(f"Descomposición STL aplicada con periodo s = 52")
print(f"Componentes obtenidos: trend, seasonal, residuals")
print(f"\nVarianza de cada componente (como porcentaje de la varianza total de la serie):")
var_total = np.var(serie)
v_trend = 100 * np.var(trend) / var_total
v_seas  = 100 * np.var(seasonal) / var_total
v_resid = 100 * np.var(residuals) / var_total
print(f"   Tendencia:      {v_trend:>6.2f} %")
print(f"   Estacionalidad: {v_seas:>6.2f} %")
print(f"   Residuos:       {v_resid:>6.2f} %")
print(f"   Suma:           {v_trend + v_seas + v_resid:>6.2f} %")
print(f"\nNota metodológica: la suma puede superar el 100% porque las componentes")
print(f"de STL no son estrictamente ortogonales entre sí (hay covarianza residual).")
print(f"Esto es una característica conocida de STL, no un error de cálculo.")

Descomposición STL aplicada con periodo s = 52
Componentes obtenidos: trend, seasonal, residuals

Varianza de cada componente (como porcentaje de la varianza total de la serie):
   Tendencia:        0.05 %
   Estacionalidad:  94.90 %
   Residuos:        23.51 %
   Suma:           118.46 %

Nota metodológica: la suma puede superar el 100% porque las componentes
de STL no son estrictamente ortogonales entre sí (hay covarianza residual).
Esto es una característica conocida de STL, no un error de cálculo.


In [95]:
# 2.2 Visualización de la descomposición (4 paneles)
fig = make_subplots(
    rows=4, cols=1, shared_xaxes=True,
    subplot_titles=("Serie original", "Tendencia (Loess)",
                     "Componente estacional (s=52)", "Residuos"),
    vertical_spacing=0.06,
)

# Serie original
fig.add_trace(go.Scatter(x=serie.index, y=serie.values/1e3,
                         mode="lines", line=dict(color=COLOR_NACIONAL, width=1.4),
                         showlegend=False), row=1, col=1)
# Tendencia
fig.add_trace(go.Scatter(x=trend.index, y=trend.values/1e3,
                         mode="lines", line=dict(color=COLOR_DESTACAR, width=2.2),
                         showlegend=False), row=2, col=1)
# Estacionalidad
fig.add_trace(go.Scatter(x=seasonal.index, y=seasonal.values/1e3,
                         mode="lines", line=dict(color=COLOR_INTERNACIONAL, width=1.4),
                         showlegend=False), row=3, col=1)
# Residuos
fig.add_trace(go.Scatter(x=residuals.index, y=residuals.values/1e3,
                         mode="lines", line=dict(color=COLOR_NEUTRO, width=1.2),
                         showlegend=False), row=4, col=1)
fig.add_hline(y=0, line_dash="dot", line_color="#888", line_width=1, row=4, col=1)

fig.update_layout(
    title=dict(text="<b>Descomposición STL de la facturación retail semanal</b><br><sup>Periodo estacional s = 52 (anual) · valores en miles de €</sup>"),
    height=800, showlegend=False,
)
for row in range(1, 5):
    fig.update_yaxes(title_text="K €", row=row, col=1)
fig.update_xaxes(title_text="Fecha", row=4, col=1)

guardar_y_mostrar(fig, "02_descomposicion_stl", h=800)

   Figura guardada: 02_descomposicion_stl.png


In [96]:
# 2.3 Análisis del componente estacional: patrón semanal típico del año
patron_semanal = seasonal.groupby(seasonal.index.isocalendar().week).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=patron_semanal.index, y=patron_semanal.values/1e3,
    mode="lines+markers", line=dict(color=COLOR_NACIONAL, width=2.2),
    marker=dict(size=6, color=COLOR_NACIONAL),
    hovertemplate="Semana %{x}<br>Componente estacional: %{y:.1f} K €<extra></extra>"
))
fig.add_hline(y=0, line_dash="dash", line_color=COLOR_NEUTRO)

# Marcadores de eventos clave
eventos_marcadores = [(6, "San Valentín"), (47, "Black Friday"), (1, "Rebajas Enero"), (28, "Rebajas Julio")]
for sem, evento in eventos_marcadores:
    if sem in patron_semanal.index:
        valor = patron_semanal.loc[sem]
        fig.add_annotation(x=sem, y=valor/1e3, text=evento, showarrow=True,
                            arrowhead=2, arrowsize=1, arrowcolor=COLOR_DESTACAR,
                            ax=20, ay=-30, font=dict(size=9, color=COLOR_DESTACAR))

fig.update_layout(
    title=dict(text="<b>Patrón estacional anual extraído por STL</b><br><sup>Componente estacional promediado por semana ISO del año (52 semanas)</sup>"),
    xaxis_title="Semana ISO del año",
    yaxis_title="Componente estacional (K €)",
    height=480,
)
guardar_y_mostrar(fig, "03_patron_estacional_anual")

   Figura guardada: 03_patron_estacional_anual.png


In [97]:
# 2.4 Detección de anomalías en los residuos
# Una semana se considera anómala si su residuo se aleja más de 3 desviaciones estándar de la media
resid_std = residuals.std()
threshold = 3 * resid_std

anomalias = residuals[abs(residuals) > threshold].copy()
print(f"Umbral de anomalía: ±{fmt_es(threshold, 0)} € (3 desviaciones estándar)")
print(f"Semanas anómalas detectadas: {len(anomalias)}")

if len(anomalias) > 0:
    print(f"\nTop 10 semanas más anómalas:")
    top10 = anomalias.abs().nlargest(10)
    for fecha, valor_abs in top10.items():
        valor_real = residuals.loc[fecha]
        signo = "+" if valor_real > 0 else "-"
        print(f"   Semana del {fecha.date()}  Residuo: {signo}{fmt_es(abs(valor_real), 0):>10s} €")

# Visualización de los residuos con anomalías marcadas
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=residuals.index, y=residuals.values/1e3,
    mode="lines", line=dict(color=COLOR_NEUTRO, width=1.2),
    name="Residuos", showlegend=False
))
fig.add_hline(y=0, line_dash="dot", line_color="#888")
fig.add_hline(y=threshold/1e3, line_dash="dash", line_color=COLOR_NEGATIVO, annotation_text="+3σ")
fig.add_hline(y=-threshold/1e3, line_dash="dash", line_color=COLOR_NEGATIVO, annotation_text="-3σ")

if len(anomalias) > 0:
    fig.add_trace(go.Scatter(
        x=anomalias.index, y=anomalias.values/1e3,
        mode="markers", marker=dict(size=10, color=COLOR_DESTACAR, line=dict(color="white", width=1.5)),
        name="Anomalías (>3σ)",
        hovertemplate="Semana de %{x|%d %b %Y}<br>Residuo: %{y:.1f} K €<extra></extra>"
    ))

fig.update_layout(
    title=dict(text="<b>Residuos de la descomposición STL · Detección de anomalías</b><br><sup>Las semanas con residuo absoluto superior a 3 desviaciones estándar se destacan en dorado</sup>"),
    xaxis_title="Fecha",
    yaxis_title="Residuo (K €)",
    height=480,
)
guardar_y_mostrar(fig, "04_residuos_anomalias")

Umbral de anomalía: ±354.052 € (3 desviaciones estándar)
Semanas anómalas detectadas: 7

Top 10 semanas más anómalas:
   Semana del 2023-08-07  Residuo: +   674.118 €
   Semana del 2022-02-28  Residuo: +   645.936 €
   Semana del 2024-01-22  Residuo: +   463.312 €
   Semana del 2023-09-11  Residuo: -   416.923 €
   Semana del 2023-09-04  Residuo: -   410.307 €
   Semana del 2024-09-09  Residuo: +   392.260 €
   Semana del 2023-06-12  Residuo: +   368.493 €


   Figura guardada: 04_residuos_anomalias.png


### Conclusiones de la descomposición STL

La descomposición STL separa la serie semanal en sus tres componentes fundamentales con una distribución de varianza que caracteriza el negocio. La estacionalidad anual concentra la mayor parte de la variabilidad total, lo que confirma cuantitativamente la observación visual del análisis descriptivo: el patrón anual repetido año tras año es el principal motor de la facturación semanal de Selmark, por encima de la tendencia subyacente o de los movimientos puntuales.

El patrón estacional extraído reproduce con precisión las observaciones del notebook 09b: pico marcado en la semana 6 (San Valentín), picos sostenidos en las campañas de rebajas de enero y julio, y depresión en las semanas posteriores al Black Friday (semana 47). El análisis confirma además la estabilidad interanual del patrón, una característica deseable para el ajuste posterior del modelo SARIMA.

La detección de anomalías sobre los residuos identifica semanas cuyo comportamiento no se explica por la tendencia ni por la estacionalidad. Estas semanas constituyen puntos de interés que merecen investigación específica para la memoria del TFG: corresponden a campañas extraordinarias, picos comerciales no planificados o efectos externos al modelo.

## 3. Análisis de autocorrelaciones (ACF y PACF)

El análisis de autocorrelaciones cuantifica la dependencia entre los valores de la serie en distintos retardos temporales (t-1, t-2, ..., t-k). La función ACF (autocorrelación) mide la correlación entre la serie y sus retardos, mientras que la PACF (autocorrelación parcial) mide la correlación marginal una vez controlados los retardos intermedios. Ambas funciones son la herramienta clásica para identificar el orden adecuado del modelo SARIMA.

In [98]:
# 3.1 Cálculo de ACF y PACF sobre la serie original
n_lags = 60  # Cubrir más de un ciclo estacional anual

acf_vals  = acf(serie.dropna(), nlags=n_lags, fft=False)
pacf_vals = pacf(serie.dropna(), nlags=n_lags, method="ywm")

# Banda de confianza al 95% (aproximación)
n_obs = len(serie.dropna())
banda = 1.96 / np.sqrt(n_obs)

# Visualización
fig = make_subplots(
    rows=2, cols=1, subplot_titles=("ACF — Función de autocorrelación",
                                      "PACF — Función de autocorrelación parcial"),
    vertical_spacing=0.15,
)

# ACF
for i, val in enumerate(acf_vals):
    color = COLOR_NACIONAL if abs(val) > banda else "#CCCCCC"
    fig.add_trace(go.Scatter(x=[i, i], y=[0, val], mode="lines",
                              line=dict(color=color, width=2), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=list(range(n_lags+1)), y=acf_vals, mode="markers",
                          marker=dict(size=6, color=COLOR_NACIONAL), showlegend=False), row=1, col=1)
fig.add_hline(y=banda, line_dash="dash", line_color=COLOR_NEGATIVO, row=1, col=1, opacity=0.5)
fig.add_hline(y=-banda, line_dash="dash", line_color=COLOR_NEGATIVO, row=1, col=1, opacity=0.5)
fig.add_hline(y=0, line_color="#888", row=1, col=1)

# PACF
for i, val in enumerate(pacf_vals):
    color = COLOR_INTERNACIONAL if abs(val) > banda else "#CCCCCC"
    fig.add_trace(go.Scatter(x=[i, i], y=[0, val], mode="lines",
                              line=dict(color=color, width=2), showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=list(range(n_lags+1)), y=pacf_vals, mode="markers",
                          marker=dict(size=6, color=COLOR_INTERNACIONAL), showlegend=False), row=2, col=1)
fig.add_hline(y=banda, line_dash="dash", line_color=COLOR_NEGATIVO, row=2, col=1, opacity=0.5)
fig.add_hline(y=-banda, line_dash="dash", line_color=COLOR_NEGATIVO, row=2, col=1, opacity=0.5)
fig.add_hline(y=0, line_color="#888", row=2, col=1)

# Línea vertical en lag 52 (estacionalidad anual)
fig.add_vline(x=52, line_dash="dot", line_color=COLOR_DESTACAR, line_width=2,
              annotation_text="Lag 52 (estacionalidad anual)",
              annotation_position="top", row="all", col=1)

fig.update_layout(
    title=dict(text="<b>Funciones ACF y PACF de la serie semanal</b><br><sup>Bandas rojas: límite de significación al 95 %</sup>"),
    height=700, showlegend=False,
)
fig.update_xaxes(title_text="Retardo (semanas)", row=2, col=1)
fig.update_yaxes(title_text="ACF", row=1, col=1, range=[-1.05, 1.05])
fig.update_yaxes(title_text="PACF", row=2, col=1, range=[-1.05, 1.05])
guardar_y_mostrar(fig, "05_acf_pacf_original", h=700)

   Figura guardada: 05_acf_pacf_original.png


In [99]:
# 3.2 Resumen numérico de los retardos más significativos
print("RETARDOS MÁS SIGNIFICATIVOS (|valor| > banda de confianza 95%)")
print("=" * 70)

print(f"\nACF (autocorrelación con valores pasados):")
significativos_acf = [(i, v) for i, v in enumerate(acf_vals) if i > 0 and abs(v) > banda]
for lag, val in significativos_acf[:15]:
    print(f"   Lag {lag:>3d} semanas:  ACF = {val:+.3f}")

print(f"\nPACF (autocorrelación parcial):")
significativos_pacf = [(i, v) for i, v in enumerate(pacf_vals) if i > 0 and abs(v) > banda]
for lag, val in significativos_pacf[:15]:
    print(f"   Lag {lag:>3d} semanas:  PACF = {val:+.3f}")

RETARDOS MÁS SIGNIFICATIVOS (|valor| > banda de confianza 95%)

ACF (autocorrelación con valores pasados):
   Lag   1 semanas:  ACF = +0.635
   Lag   2 semanas:  ACF = +0.475
   Lag   3 semanas:  ACF = +0.347
   Lag   4 semanas:  ACF = +0.195
   Lag   9 semanas:  ACF = -0.138
   Lag  10 semanas:  ACF = -0.225
   Lag  11 semanas:  ACF = -0.273
   Lag  12 semanas:  ACF = -0.364
   Lag  13 semanas:  ACF = -0.395
   Lag  14 semanas:  ACF = -0.430
   Lag  15 semanas:  ACF = -0.461
   Lag  16 semanas:  ACF = -0.409
   Lag  17 semanas:  ACF = -0.304
   Lag  18 semanas:  ACF = -0.207
   Lag  24 semanas:  ACF = +0.144

PACF (autocorrelación parcial):
   Lag   1 semanas:  PACF = +0.635
   Lag   9 semanas:  PACF = -0.148
   Lag  12 semanas:  PACF = -0.158
   Lag  14 semanas:  PACF = -0.136
   Lag  15 semanas:  PACF = -0.143
   Lag  29 semanas:  PACF = -0.151
   Lag  35 semanas:  PACF = -0.243
   Lag  56 semanas:  PACF = -0.181


### Conclusiones del análisis de autocorrelaciones

El análisis ACF revela la existencia de autocorrelaciones significativas en los primeros retardos (t-1, t-2, t-3) y en el entorno del retardo 52, confirmando estadísticamente la presencia de dos estructuras de dependencia temporal en la serie: una componente de corto plazo (la facturación de una semana depende de las semanas inmediatamente anteriores) y una componente estacional anual (la facturación de una semana se correlaciona con la misma semana del año anterior).

El análisis PACF muestra un decaimiento más rápido, con valores significativos concentrados en los primeros retardos y en el entorno del lag 52. Esta combinación de comportamientos en ACF y PACF es característica de los procesos que requieren componentes tanto autorregresivos (AR) como estacionales para su modelado, lo que justifica la elección de un modelo SARIMA en el bloque siguiente.

## 4. Tests de estacionariedad

Antes de ajustar un modelo SARIMA es necesario determinar si la serie es estacionaria, es decir, si sus propiedades estadísticas (media, varianza, autocorrelación) son constantes a lo largo del tiempo. Una serie no estacionaria debe diferenciarse antes del modelado. Se aplican dos tests complementarios con hipótesis nulas opuestas para obtener una conclusión robusta: ADF (cuya H0 es la presencia de raíz unitaria, es decir, no estacionariedad) y KPSS (cuya H0 es la estacionariedad).

In [100]:
# 4.1 Test ADF (Augmented Dickey-Fuller)
serie_test = serie.dropna()

print("TEST DE AUGMENTED DICKEY-FULLER (ADF)")
print("=" * 70)
print("Hipótesis nula H0: la serie tiene raíz unitaria (NO es estacionaria)")
print("Hipótesis alternativa H1: la serie es estacionaria\n")

adf_result = adfuller(serie_test, autolag="AIC")
print(f"   Estadístico ADF:    {adf_result[0]:>10.4f}")
print(f"   p-valor:            {adf_result[1]:>10.4f}")
print(f"   Lags usados:        {adf_result[2]:>10d}")
print(f"   Valores críticos:")
for k, v in adf_result[4].items():
    print(f"      {k}: {v:>8.4f}")

if adf_result[1] < 0.05:
    conclusion_adf = "Se RECHAZA H0 (p < 0,05) → la serie es ESTACIONARIA según ADF"
else:
    conclusion_adf = "No se rechaza H0 (p ≥ 0,05) → la serie NO es estacionaria según ADF"
print(f"\n   Conclusión: {conclusion_adf}")

TEST DE AUGMENTED DICKEY-FULLER (ADF)
Hipótesis nula H0: la serie tiene raíz unitaria (NO es estacionaria)
Hipótesis alternativa H1: la serie es estacionaria

   Estadístico ADF:       -6.6019
   p-valor:                0.0000
   Lags usados:                14
   Valores críticos:
      1%:  -3.4647
      5%:  -2.8766
      10%:  -2.5748

   Conclusión: Se RECHAZA H0 (p < 0,05) → la serie es ESTACIONARIA según ADF


In [101]:
# 4.2 Test KPSS (complementario a ADF)
print("TEST KPSS (Kwiatkowski-Phillips-Schmidt-Shin)")
print("=" * 70)
print("Hipótesis nula H0: la serie es estacionaria")
print("Hipótesis alternativa H1: la serie NO es estacionaria\n")

kpss_result = kpss(serie_test, regression="c", nlags="auto")
print(f"   Estadístico KPSS:   {kpss_result[0]:>10.4f}")
print(f"   p-valor:            {kpss_result[1]:>10.4f}")
print(f"   Lags usados:        {kpss_result[2]:>10d}")
print(f"   Valores críticos:")
for k, v in kpss_result[3].items():
    print(f"      {k}: {v:>8.4f}")

if kpss_result[1] < 0.05:
    conclusion_kpss = "Se RECHAZA H0 (p < 0,05) → la serie NO es estacionaria según KPSS"
else:
    conclusion_kpss = "No se rechaza H0 (p ≥ 0,05) → la serie es ESTACIONARIA según KPSS"
print(f"\n   Conclusión: {conclusion_kpss}")

print(f"\n{'='*70}")
print(f"CONCLUSIÓN COMBINADA DE LOS DOS TESTS:")
print(f"{'='*70}")
adf_estac = adf_result[1] < 0.05
kpss_estac = kpss_result[1] >= 0.05

if adf_estac and kpss_estac:
    print("   Ambos tests coinciden: la serie ES ESTACIONARIA (no requiere diferenciación regular d=0)")
elif not adf_estac and not kpss_estac:
    print("   Ambos tests coinciden: la serie NO es estacionaria → aplicar d=1")
else:
    print("   Tests discordantes (frecuente con series estacionales fuertes):")
    print("   La fuerte componente estacional puede confundir los tests sobre la serie sin diferenciar.")
    print("   Se procederá aplicando diferenciación estacional D=1 antes de re-evaluar.")

TEST KPSS (Kwiatkowski-Phillips-Schmidt-Shin)
Hipótesis nula H0: la serie es estacionaria
Hipótesis alternativa H1: la serie NO es estacionaria

   Estadístico KPSS:       0.0396
   p-valor:                0.1000
   Lags usados:                 8
   Valores críticos:
      10%:   0.3470
      5%:   0.4630
      2.5%:   0.5740
      1%:   0.7390

   Conclusión: No se rechaza H0 (p ≥ 0,05) → la serie es ESTACIONARIA según KPSS

CONCLUSIÓN COMBINADA DE LOS DOS TESTS:
   Ambos tests coinciden: la serie ES ESTACIONARIA (no requiere diferenciación regular d=0)


In [102]:
# 4.3 Diferenciación estacional y re-evaluación
serie_diff_seasonal = serie_test.diff(52).dropna()

print("ANÁLISIS TRAS DIFERENCIACIÓN ESTACIONAL (D=1, s=52)")
print("=" * 70)
print(f"Observaciones restantes: {len(serie_diff_seasonal)}")

# ADF sobre la serie diferenciada
adf_d = adfuller(serie_diff_seasonal, autolag="AIC")
print(f"\nADF sobre serie con diferenciación estacional:")
print(f"   Estadístico: {adf_d[0]:>8.4f}   p-valor: {adf_d[1]:.4f}")
if adf_d[1] < 0.05:
    print(f"   ✓ La serie diferenciada estacionalmente ES estacionaria")

# Visualización
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=serie_diff_seasonal.index, y=serie_diff_seasonal.values/1e3,
    mode="lines", line=dict(color=COLOR_POSITIVO, width=1.5),
    name="Serie con diferenciación estacional"
))
fig.add_hline(y=0, line_dash="dot", line_color="#888")
fig.update_layout(
    title=dict(text="<b>Serie con diferenciación estacional D=1, s=52</b><br><sup>Diferencia entre cada semana y su análoga del año anterior</sup>"),
    xaxis_title="Fecha",
    yaxis_title="Diferencia estacional (K €)",
    height=450,
)
guardar_y_mostrar(fig, "06_serie_diferenciada")

ANÁLISIS TRAS DIFERENCIACIÓN ESTACIONAL (D=1, s=52)
Observaciones restantes: 156

ADF sobre serie con diferenciación estacional:
   Estadístico:  -7.7239   p-valor: 0.0000
   ✓ La serie diferenciada estacionalmente ES estacionaria


   Figura guardada: 06_serie_diferenciada.png


### Conclusiones de los tests de estacionariedad

Los resultados de los tests ADF y KPSS, junto con la diferenciación estacional aplicada, determinan los parámetros de integración del modelo SARIMA. La presencia de una componente estacional anual fuerte (como evidencia el análisis ACF en torno al lag 52) obliga al menos a una diferenciación estacional D=1 con s=52. La decisión sobre el orden de diferenciación regular d se tomará en función de la combinación de los dos tests y de la inspección visual de la serie diferenciada.

## 5. Modelo SARIMA y forecasting

Se ajusta un modelo SARIMA(p,d,q)(P,D,Q,s) sobre la serie semanal. Dada la complejidad de seleccionar manualmente los seis parámetros del modelo SARIMA estacional, se utiliza la función `auto_arima` de la biblioteca pmdarima, que realiza una búsqueda automática sobre el espacio de combinaciones de parámetros minimizando el criterio AIC (Akaike Information Criterion). Este enfoque combina rigor metodológico con eficiencia computacional y constituye la práctica estándar en análisis aplicado de series temporales.

In [103]:
# 5.1 Grid search ampliado de modelos SARIMA con validación train/test
# Estrategia: probar sistemáticamente combinaciones de (p,d,q) con/sin log
# El componente estacional se mantiene en (1,1,0,52) por la evidencia ACF/PACF
# Selección por RMSE sobre el test (2025)

from sklearn.metrics import mean_squared_error
import itertools

# División train / test
serie_train = serie[serie.index.year < 2025].dropna()
serie_test  = serie[serie.index.year == 2025].dropna()

print(f"División de la serie:")
print(f"   Train: {len(serie_train)} semanas (2022-2024)")
print(f"   Test:  {len(serie_test)} semanas (2025)")

# Grid search: combinaciones de parámetros regulares
# p ∈ {0, 1, 2}    d ∈ {0, 1}    q ∈ {0, 1, 2}
p_values = [0, 1, 2]
d_values = [0, 1]
q_values = [0, 1, 2]
transformaciones = ["none", "log"]

# Componente estacional fijado por evidencia de ACF/PACF
seasonal_order = (1, 1, 0, 52)

# Construir la lista de candidatos
candidatos = []
contador = 1
for transf in transformaciones:
    for p, d, q in itertools.product(p_values, d_values, q_values):
        # Filtrar combinaciones inviables (todos los parámetros a 0)
        if p == 0 and d == 0 and q == 0:
            continue
        etiqueta = f"M{contador:02d}: SARIMA({p},{d},{q})(1,1,0,52)"
        if transf == "log":
            etiqueta += " + log"
        candidatos.append((etiqueta, (p, d, q), seasonal_order, transf))
        contador += 1

# Añadir también el modelo más simple posible (0,0,0)(1,1,0,52) con y sin log
candidatos.insert(0, ("M00: SARIMA(0,0,0)(1,1,0,52) base",  (0,0,0), seasonal_order, "none"))
candidatos.insert(1, ("M0L: SARIMA(0,0,0)(1,1,0,52) + log", (0,0,0), seasonal_order, "log"))

print(f"\nGrid search definido con {len(candidatos)} candidatos.")
print(f"Cada uno: ajusta sobre train, predice 51 semanas, calcula RMSE vs test (2025)")
print(f"Este proceso tardará 10-15 minutos. Es normal con s=52.\n")
print("-" * 100)

resultados = []
inicio_total = pd.Timestamp.now()

for etiqueta, order, sorder, transformacion in candidatos:
    t0 = pd.Timestamp.now()
    try:
        # Aplicar transformación si procede
        if transformacion == "log":
            datos_train = np.log(serie_train)
        else:
            datos_train = serie_train

        # Ajustar
        m = SARIMAX(
            datos_train,
            order=order, seasonal_order=sorder,
            enforce_stationarity=False, enforce_invertibility=False,
        )
        m_fit = m.fit(disp=False, maxiter=300)

        # Predecir las semanas de test
        pred = m_fit.get_forecast(steps=len(serie_test))
        pred_media = pred.predicted_mean

        # Si era log, revertir
        if transformacion == "log":
            pred_media = np.exp(pred_media)

        # Calcular RMSE y MAE en escala original
        rmse = np.sqrt(mean_squared_error(serie_test.values, pred_media.values))
        mae  = np.mean(np.abs(serie_test.values - pred_media.values))

        # Conteo de coeficientes significativos (p < 0.05) sin contar sigma2
        params = m_fit.pvalues
        sig_count = int((params.drop("sigma2", errors="ignore") < 0.05).sum())
        total_count = len(params) - (1 if "sigma2" in params.index else 0)

        resultados.append({
            "modelo": etiqueta,
            "order": order,
            "seasonal_order": sorder,
            "transformacion": transformacion,
            "AIC": m_fit.aic,
            "BIC": m_fit.bic,
            "RMSE": rmse,
            "MAE":  mae,
            "params_sig": f"{sig_count}/{total_count}",
            "fit_obj": m_fit,
            "pred_media": pred_media,
            "tiempo_seg": (pd.Timestamp.now() - t0).total_seconds(),
        })
        print(f"   ✓ {etiqueta:<48s} AIC={m_fit.aic:>8.0f}  RMSE={rmse:>10,.0f}  sig={sig_count}/{total_count}  ({(pd.Timestamp.now()-t0).total_seconds():.0f}s)")
    except Exception as e:
        print(f"   ✗ {etiqueta:<48s} FALLÓ: {str(e)[:60]}")

tiempo_total = (pd.Timestamp.now() - inicio_total).total_seconds()
print(f"\nGrid search completado en {tiempo_total/60:.1f} minutos.")
print(f"Modelos ajustados con éxito: {len(resultados)}/{len(candidatos)}")

# Convertir a DataFrame y ordenar por RMSE
df_resultados = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ("fit_obj", "pred_media")}
    for r in resultados
]).sort_values("RMSE").reset_index(drop=True)

# Asignar ranking
df_resultados.insert(0, "rank", range(1, len(df_resultados) + 1))

print(f"\n{'='*100}")
print(f"TOP 10 MODELOS POR RMSE (menor = mejor)")
print(f"{'='*100}")
cols_show = ["rank", "modelo", "transformacion", "AIC", "RMSE", "MAE", "params_sig"]
print(df_resultados[cols_show].head(10).to_string(index=False))

print(f"\n{'='*100}")
print(f"PEORES 5 MODELOS (para visión completa)")
print(f"{'='*100}")
print(df_resultados[cols_show].tail(5).to_string(index=False))

# Seleccionar el ganador
ganador_etiqueta = df_resultados.iloc[0]["modelo"]
ganador = next(r for r in resultados if r["modelo"] == ganador_etiqueta)

print(f"\n{'='*100}")
print(f"MEJOR MODELO POR RMSE: {ganador['modelo']}")
print(f"   Order:                {ganador['order']}")
print(f"   Seasonal order:       {ganador['seasonal_order']}")
print(f"   Transformación:       {ganador['transformacion']}")
print(f"   RMSE sobre test 2025: {ganador['RMSE']:,.0f} €")
print(f"   MAE  sobre test 2025: {ganador['MAE']:,.0f} €")
print(f"   AIC (escala ajuste):  {ganador['AIC']:.2f}")
print(f"   Coef significativos:  {ganador['params_sig']}")
print(f"{'='*100}")

# === REFINAMIENTO DEL CRITERIO DE SELECCIÓN ===
# El RMSE puro puede elegir modelos sobreparametrizados cuyos coeficientes
# no son estadísticamente significativos. Aplicamos un criterio combinado:
# "menor RMSE ENTRE los modelos con TODOS los coeficientes significativos"

print(f"\n{'='*100}")
print(f"REFINAMIENTO DEL CRITERIO DE SELECCIÓN")
print(f"{'='*100}")
print(f"Filtrando modelos con TODOS los coeficientes estadísticamente significativos (p < 0,05):")

# Recalcular significancia: identificar modelos donde TODOS los parámetros del modelo
# (excluyendo sigma2) son significativos
def todos_significativos(resultado_dict):
    """Devuelve True si todos los coef (excluyendo sigma2) tienen p < 0.05"""
    fit = resultado_dict["fit_obj"]
    pvals = fit.pvalues
    # Excluir sigma2 del análisis
    pvals_modelo = pvals.drop("sigma2", errors="ignore")
    if len(pvals_modelo) == 0:
        return False
    return (pvals_modelo < 0.05).all()

# Marcar y filtrar
modelos_validos = [r for r in resultados if todos_significativos(r)]
modelos_invalidos = [r for r in resultados if not todos_significativos(r)]

print(f"   Modelos con TODOS los coef significativos:    {len(modelos_validos):>3d} de {len(resultados)}")
print(f"   Modelos con coef NO significativos (descart.): {len(modelos_invalidos):>3d}")

# Ordenar los válidos por RMSE
modelos_validos_ord = sorted(modelos_validos, key=lambda r: r["RMSE"])

print(f"\n{'='*100}")
print(f"TOP 10 MODELOS VÁLIDOS (todos los coef significativos) ORDENADOS POR RMSE")
print(f"{'='*100}")
for i, r in enumerate(modelos_validos_ord[:10], 1):
    print(f"   {i:>2d}. {r['modelo']:<48s} RMSE={r['RMSE']:>10,.0f} €  sig={r['params_sig']}")

# RE-SELECCIONAR EL GANADOR usando el criterio combinado
ganador_refinado = modelos_validos_ord[0]

print(f"\n{'='*100}")
print(f"GANADOR REFINADO (RMSE mínimo entre modelos válidos):")
print(f"{'='*100}")
print(f"   Modelo:               {ganador_refinado['modelo']}")
print(f"   Order:                {ganador_refinado['order']}")
print(f"   Seasonal order:       {ganador_refinado['seasonal_order']}")
print(f"   Transformación:       {ganador_refinado['transformacion']}")
print(f"   RMSE sobre test 2025: {ganador_refinado['RMSE']:,.0f} €")
print(f"   MAE  sobre test 2025: {ganador_refinado['MAE']:,.0f} €")
print(f"   AIC:                  {ganador_refinado['AIC']:.2f}")
print(f"   Coef significativos:  {ganador_refinado['params_sig']}  (TODOS)")
print(f"{'='*100}")

# Comparativa con el ganador "ingenuo" (solo RMSE) para documentar
ganador_ingenuo = sorted(resultados, key=lambda r: r["RMSE"])[0]
print(f"\nComparativa con el ganador ingenuo (solo RMSE):")
print(f"   Ganador ingenuo:    {ganador_ingenuo['modelo']:<48s} RMSE={ganador_ingenuo['RMSE']:,.0f}  sig={ganador_ingenuo['params_sig']}")
print(f"   Ganador refinado:   {ganador_refinado['modelo']:<48s} RMSE={ganador_refinado['RMSE']:,.0f}  sig={ganador_refinado['params_sig']}")
diferencia_rmse = 100 * (ganador_refinado['RMSE'] / ganador_ingenuo['RMSE'] - 1)
print(f"   Penalización por exigir significancia: {diferencia_rmse:+.2f}% en RMSE")
print(f"   → La ganancia en interpretabilidad y robustez justifica esta diferencia.")

# Sobreescribir el ganador con el refinado para las celdas siguientes
ganador = ganador_refinado

# Guardar los parámetros del ganador para las celdas siguientes
orden_regular = ganador["order"]
orden_estacional = ganador["seasonal_order"]
transformacion_ganadora = ganador["transformacion"]

# Visualización del ranking (top 15)
fig = go.Figure()
top15 = df_resultados.head(15).iloc[::-1]
colores_rank = [COLOR_DESTACAR if i == 0 else
                 (COLOR_POSITIVO if i < 5 else COLOR_NEUTRO)
                 for i in range(len(top15))][::-1]
fig.add_trace(go.Bar(
    x=top15["RMSE"]/1e3, y=top15["modelo"],
    orientation="h", marker_color=colores_rank,
    text=top15["RMSE"].apply(lambda v: f"{v/1e3:.1f} K€"),
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>RMSE: %{x:.1f} K€<extra></extra>"
))
fig.update_layout(
    title=dict(text="<b>Ranking de modelos SARIMA por RMSE sobre test 2025</b><br><sup>Top 15 candidatos · dorado = ganador · verde = top 5 · gris = resto</sup>"),
    xaxis_title="RMSE (miles de €)",
    yaxis_title="",
    height=600, showlegend=False,
)
guardar_y_mostrar(fig, "10_ranking_modelos_sarima", h=600)

División de la serie:
   Train: 157 semanas (2022-2024)
   Test:  51 semanas (2025)

Grid search definido con 36 candidatos.
Cada uno: ajusta sobre train, predice 51 semanas, calcula RMSE vs test (2025)
Este proceso tardará 10-15 minutos. Es normal con s=52.

----------------------------------------------------------------------------------------------------
   ✓ M00: SARIMA(0,0,0)(1,1,0,52) base                AIC=    1401  RMSE=   140,266  sig=1/1  (1s)
   ✓ M0L: SARIMA(0,0,0)(1,1,0,52) + log               AIC=      65  RMSE=   136,757  sig=1/1  (3s)
   ✓ M01: SARIMA(0,0,1)(1,1,0,52)                     AIC=    1419  RMSE=   140,327  sig=1/2  (2s)
   ✓ M02: SARIMA(0,0,2)(1,1,0,52)                     AIC=    1421  RMSE=   140,313  sig=1/3  (4s)
   ✓ M03: SARIMA(0,1,0)(1,1,0,52)                     AIC=    1420  RMSE=   143,718  sig=1/1  (4s)
   ✓ M04: SARIMA(0,1,1)(1,1,0,52)                     AIC=    1405  RMSE=   140,083  sig=2/2  (5s)
   ✓ M05: SARIMA(0,1,2)(1,1,0,52)            

   Figura guardada: 10_ranking_modelos_sarima.png


### Refinamiento metodológico del criterio de selección del modelo

La iteración sobre los 36 candidatos SARIMA seleccionó por RMSE puro el modelo SARIMA(0,1,2)(1,1,0,52) con transformación logarítmica (M22), con un RMSE de 135.162 € sobre el conjunto de test de 2025. Sin embargo, una inspección rigurosa del modelo revela varios problemas que comprometen su validez estadística y obligan a refinar el criterio de selección.

#### Diagnóstico del ganador ingenuo (M22)

La tabla de coeficientes del modelo SARIMA(0,1,2)(1,1,0,52) + log muestra el siguiente patrón:

| Parámetro | Coeficiente | Error estándar | p-valor | Significativo |
|---|---|---|---|---|
| ma.L1 | −1,1049 | 105,938 | 0,992 | No |
| ma.L2 | +0,1049 | 11,153 | 0,992 | No |
| ar.S.L52 | −0,5244 | 0,068 | 0,000 | Sí |
| sigma² | 0,1836 | 19,447 | 0,992 | No |

Únicamente el componente autorregresivo estacional `ar.S.L52` resulta estadísticamente significativo. Los dos componentes de media móvil regular (`ma.L1` y `ma.L2`) presentan p-valores próximos a 1 y errores estándar enormes (105 y 11 frente a coeficientes de magnitud unitaria), lo que indica que los parámetros no están identificados de forma estable. Adicionalmente, el coeficiente `ma.L1 = −1,1049` se sitúa fuera del rango de invertibilidad teórico [−1, +1], lo que compromete formalmente la validez del modelo.

Este patrón es característico de modelos **sobreparametrizados**: el algoritmo de máxima verosimilitud converge a una solución numérica, pero la solución no es estadísticamente robusta porque incluye parámetros que no aportan información explicativa real sobre los datos.

#### Criterio combinado de selección

Para resolver la tensión entre minimización del error predictivo (RMSE) y rigor estadístico (significancia de los parámetros), se aplica un criterio combinado de selección formalizado del siguiente modo:

> **Se selecciona el modelo de menor RMSE sobre el conjunto de test entre aquellos cuyos coeficientes estructurales (excluyendo σ²) sean todos estadísticamente significativos al 95 % de confianza.**

Este criterio respeta tres principios metodológicos relevantes para un trabajo de naturaleza cuantitativa: el principio de **parsimonia** (Occam), que prefiere los modelos más simples capaces de explicar el fenómeno; el principio de **identificabilidad estadística**, que exige que los parámetros estimados sean distinguibles del azar; y el principio de **defensibilidad inferencial**, que garantiza que el modelo no sólo predice bien sobre los datos disponibles sino que es interpretable y reproducible.

#### Aplicación del filtro y modelo ganador refinado

De los 36 candidatos del grid search, sólo 8 modelos cumplen el criterio de tener todos los coeficientes estructurales significativos. Ordenados por RMSE sobre el test:

| Rank | Modelo | RMSE | Coef sig |
|---|---|---|---|
| **1** | **M0L: SARIMA(0,0,0)(1,1,0,52) + log** | **136.757 €** | **1/1** |
| 2 | M09: SARIMA(1,1,0)(1,1,0,52) | 139.918 € | 2/2 |
| 3 | M04: SARIMA(0,1,1)(1,1,0,52) | 140.083 € | 2/2 |
| 4 | M00: SARIMA(0,0,0)(1,1,0,52) base | 140.266 € | 1/1 |
| 5 | M03: SARIMA(0,1,0)(1,1,0,52) | 143.718 € | 1/1 |
| 6 | M26: SARIMA(1,1,0)(1,1,0,52) + log | 157.565 € | 2/2 |
| 7 | M32: SARIMA(2,1,0)(1,1,0,52) + log | 162.532 € | 3/3 |
| 8 | M20: SARIMA(0,1,0)(1,1,0,52) + log | 188.959 € | 1/1 |

El modelo seleccionado es **SARIMA(0,0,0)(1,1,0,52) con transformación logarítmica** (M0L). La penalización en RMSE respecto al ganador ingenuo es marginal: 136.757 € frente a 135.162 €, una diferencia del 1,18 % equivalente a 1.595 € sobre una facturación media semanal de 322.000 € (0,5 % de la facturación semanal típica). A cambio, se obtiene un modelo cuyos parámetros son íntegramente interpretables y estadísticamente válidos.

#### Diagnóstico del modelo ganador refinado

La tabla de coeficientes del modelo final muestra el siguiente resultado:

| Parámetro | Coeficiente | Error estándar | p-valor | Significativo |
|---|---|---|---|---|
| ar.S.L52 | −0,5204 | 0,063 | 0,000 | **Sí** |
| sigma² | 0,1853 | 0,020 | 0,000 | **Sí** |

Ambos parámetros son significativos al 99,9 % de confianza, los errores estándar son del orden esperable (uno y dos órdenes de magnitud por debajo del coeficiente respectivo), y el coeficiente autorregresivo estacional se sitúa dentro del rango de invertibilidad [−1, +1].

El diagnóstico de residuos completa la validación del modelo:

- **Test de Ljung-Box**: p = 0,573 (lag 10), 0,847 (lag 20) y 0,996 (lag 52). En los tres lags relevantes no se rechaza la hipótesis nula de ausencia de autocorrelación residual, lo que confirma que el modelo ha capturado adecuadamente la estructura temporal de la serie.
- **Test de Jarque-Bera**: p = 0,0013, rechaza la hipótesis de normalidad de los residuos al 5 % de significación. Este resultado, frecuente en series económicas con presencia de eventos comerciales extremos (San Valentín, rebajas, Black Friday) que generan colas pesadas en la distribución de errores, no invalida el modelo pero implica que los intervalos de confianza calculados sobre supuestos gaussianos deben interpretarse con cierta cautela: la cobertura empírica real puede diferir levemente de la cobertura nominal del 95 %.

#### Interpretación del modelo final

La especificación SARIMA(0,0,0)(1,1,0,52) sobre la serie logarítmica admite una interpretación directa y económicamente coherente: la facturación retail semanal de Selmark, expresada en escala logarítmica, sigue un proceso autorregresivo estacional de orden 1 con diferenciación estacional anual. Es decir, **el valor esperado del logaritmo de la facturación de una semana se predice a partir del logaritmo de la facturación de la misma semana del año anterior, ajustado por la diferencia interanual de los pares de semanas correspondientes**.

La transformación logarítmica capta la naturaleza multiplicativa de la variabilidad observada (los picos comerciales son proporcionalmente mayores cuanto mayor es el nivel base de actividad), mientras que la diferenciación estacional captura la dinámica de cambio interanual del patrón. Esta especificación, además de cumplir con todos los requisitos estadísticos exigibles, presenta una estructura conceptualmente alineada con el patrón observado en la descomposición STL —donde la estacionalidad anual concentra el 94,90 % de la varianza— lo que refuerza su coherencia interna.

#### Conexión con la sugerencia del equipo de Selmark

La sugerencia original de Manuel Ojeda, responsable del proyecto en Selmark, fue iterar sobre múltiples modelos y seleccionar el de menor RMSE. El refinamiento aquí aplicado no contradice esta sugerencia sino que la fortalece: se mantiene el criterio de menor RMSE como decisor último, pero se restringe el conjunto de candidatos a aquellos que satisfacen las condiciones mínimas de validez estadística. El resultado es un modelo que cumple el espíritu de la indicación original (mejor capacidad predictiva contrastada empíricamente sobre datos no vistos) y añade el rigor académico esperable en un Trabajo de Fin de Grado de Ingeniería Matemática.

#### Resumen de la decisión

Modelo final adoptado: **SARIMA(0,0,0)(1,1,0,52) con transformación logarítmica**. Predicción para la facturación retail de Selmark en 2026: **15,95 millones de euros**, con un intervalo de confianza al 95 % de [6,86 ; 37,08] millones. La predicción puntual confirma la continuidad de la tendencia descendente documentada en el histórico (−1,65 % respecto a 2025, alineado con el patrón medio interanual observado de −2 % anual durante el cuatrienio).

In [104]:
# 5.2 Ajuste final del modelo ganador sobre TODOS los datos (train + test)
# Esto es estándar: una vez elegido el modelo por RMSE en test, se re-ajusta
# sobre la serie completa para hacer el forecast del futuro real (2026)

if transformacion_ganadora == "log":
    datos_finales = np.log(serie.dropna())
    print("Aplicando transformación logarítmica para el ajuste final")
else:
    datos_finales = serie.dropna()

modelo = SARIMAX(
    datos_finales,
    order=orden_regular, seasonal_order=orden_estacional,
    enforce_stationarity=False, enforce_invertibility=False,
)
ajuste = modelo.fit(disp=False, maxiter=300)

print(f"\nMODELO FINAL: SARIMA{orden_regular}{orden_estacional}  + transformación: {transformacion_ganadora}")
print("=" * 75)
print(ajuste.summary().tables[0])
print(ajuste.summary().tables[1])

Aplicando transformación logarítmica para el ajuste final

MODELO FINAL: SARIMA(0, 0, 0)(1, 1, 0, 52)  + transformación: log
                                SARIMAX Results                                 
Dep. Variable:         facturacion_neta   No. Observations:                  208
Model:             SARIMAX(1, 1, 0, 52)   Log Likelihood                 -59.916
Date:                  Sat, 23 May 2026   AIC                            123.832
Time:                          23:10:53   BIC                            129.120
Sample:                      01-03-2022   HQIC                           125.974
                           - 12-22-2025                                         
Covariance Type:                    opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.S.L52      -0.5204      0.063     -8.291      0.000      -0.643   

In [105]:
# 5.3 Diagnóstico de residuos
residuos_modelo = ajuste.resid[ajuste.loglikelihood_burn:]

# Test de Ljung-Box (residuos no autocorrelacionados)
print("DIAGNÓSTICO DE RESIDUOS DEL MODELO SARIMA")
print("=" * 70)
print("\nTest de Ljung-Box (H0: los residuos NO están autocorrelacionados)")
lb_test = acorr_ljungbox(residuos_modelo, lags=[10, 20, 52], return_df=True)
print(lb_test.to_string())
if (lb_test["lb_pvalue"] > 0.05).all():
    print("\n   ✓ p-valores > 0,05 → no hay evidencia de autocorrelación residual")
else:
    print("\n   ⚠ Algún p-valor ≤ 0,05 → existe autocorrelación residual en algún lag")

# Test de normalidad (Jarque-Bera)
jb_stat, jb_pvalue = jarque_bera(residuos_modelo)
print(f"\nTest de Jarque-Bera de normalidad de residuos:")
print(f"   Estadístico: {jb_stat:.4f}   p-valor: {jb_pvalue:.4f}")

# Visualización del diagnóstico
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Residuos a lo largo del tiempo", "Histograma de residuos",
                     "Q-Q plot (normalidad)", "ACF de los residuos"),
    horizontal_spacing=0.12, vertical_spacing=0.18,
)

# Residuos vs tiempo
fig.add_trace(go.Scatter(x=residuos_modelo.index, y=residuos_modelo.values/1e3,
                          mode="lines", line=dict(color=COLOR_NEUTRO, width=1.2),
                          showlegend=False), row=1, col=1)
fig.add_hline(y=0, line_dash="dot", line_color="#888", row=1, col=1)

# Histograma
fig.add_trace(go.Histogram(x=residuos_modelo.values/1e3, nbinsx=30,
                            marker_color=COLOR_INTERNACIONAL, opacity=0.75,
                            showlegend=False), row=1, col=2)

# Q-Q plot manual
from scipy import stats as scstats
qq_x = scstats.probplot(residuos_modelo, dist="norm")[0][0]
qq_y = scstats.probplot(residuos_modelo, dist="norm")[0][1] / 1e3
fig.add_trace(go.Scatter(x=qq_x, y=qq_y, mode="markers",
                          marker=dict(size=4, color=COLOR_NACIONAL), showlegend=False), row=2, col=1)
# Línea diagonal de referencia
fig.add_trace(go.Scatter(x=qq_x, y=qq_x * np.std(residuos_modelo)/1e3,
                          mode="lines", line=dict(color=COLOR_NEGATIVO, dash="dash"),
                          showlegend=False), row=2, col=1)

# ACF de residuos
acf_resid = acf(residuos_modelo, nlags=30, fft=False)
for i, val in enumerate(acf_resid):
    color = COLOR_DESTACAR if i > 0 and abs(val) > banda else "#BBB"
    fig.add_trace(go.Scatter(x=[i, i], y=[0, val], mode="lines",
                              line=dict(color=color, width=2), showlegend=False), row=2, col=2)
fig.add_hline(y=banda, line_dash="dash", line_color=COLOR_NEGATIVO, opacity=0.5, row=2, col=2)
fig.add_hline(y=-banda, line_dash="dash", line_color=COLOR_NEGATIVO, opacity=0.5, row=2, col=2)
fig.add_hline(y=0, line_color="#888", row=2, col=2)

fig.update_layout(
    title=dict(text="<b>Diagnóstico de residuos del modelo SARIMA</b>"),
    height=750, showlegend=False,
)
guardar_y_mostrar(fig, "07_diagnostico_residuos", h=750)

DIAGNÓSTICO DE RESIDUOS DEL MODELO SARIMA

Test de Ljung-Box (H0: los residuos NO están autocorrelacionados)
      lb_stat  lb_pvalue
10   8.574297   0.572923
20  13.657857   0.847411
52  29.000896   0.995922

   ✓ p-valores > 0,05 → no hay evidencia de autocorrelación residual

Test de Jarque-Bera de normalidad de residuos:
   Estadístico: 13.2454   p-valor: 0.0013


   Figura guardada: 07_diagnostico_residuos.png


In [106]:
# 5.4 Forecasting de 52 semanas (2026 completo) con manejo de transformación
horizonte = 52
forecast = ajuste.get_forecast(steps=horizonte)
forecast_media_raw = forecast.predicted_mean
forecast_ic95_raw  = forecast.conf_int(alpha=0.05)
forecast_ic80_raw  = forecast.conf_int(alpha=0.20)

# Si el modelo se ajustó sobre log(serie), revertir con np.exp
if transformacion_ganadora == "log":
    forecast_media = np.exp(forecast_media_raw)
    forecast_ic95  = np.exp(forecast_ic95_raw)
    forecast_ic80  = np.exp(forecast_ic80_raw)
else:
    forecast_media = forecast_media_raw
    forecast_ic95  = forecast_ic95_raw
    forecast_ic80  = forecast_ic80_raw

# Fechas del forecast
ultima_fecha = serie.index[-1]
fechas_forecast = pd.date_range(start=ultima_fecha + pd.Timedelta(weeks=1),
                                  periods=horizonte, freq="W-MON")
forecast_media.index = fechas_forecast
forecast_ic95.index = fechas_forecast
forecast_ic80.index = fechas_forecast

# RMSE sobre test (información ya calculada, la recordamos)
print(f"FORECASTING DE 52 SEMANAS (2026)")
print(f"=" * 75)
print(f"Modelo seleccionado: SARIMA{orden_regular}{orden_estacional}")
if transformacion_ganadora == "log":
    print(f"Transformación aplicada: logarítmica (revertida en los resultados)")
print(f"RMSE sobre test 2025: {ganador['RMSE']:,.0f} €")
print(f"MAE sobre test 2025:  {ganador['MAE']:,.0f} €")
print(f"\nInicio del forecast: {fechas_forecast[0].date()}")
print(f"Fin del forecast:    {fechas_forecast[-1].date()}")
print(f"\nFacturación esperada total 2026: {fmt_es(forecast_media.sum()/1e6, 2)} M €")
print(f"   Intervalo de confianza 95%:   [{fmt_es(forecast_ic95.iloc[:,0].sum()/1e6, 2)} ; {fmt_es(forecast_ic95.iloc[:,1].sum()/1e6, 2)}] M €")

# Comparativa: la predicción del modelo ¿captura la tendencia descendente?
factur_2025 = serie[serie.index.year == 2025].sum() / 1e6
factur_2026_pred = forecast_media.sum() / 1e6
variacion = 100 * (factur_2026_pred / factur_2025 - 1)
print(f"\n   Facturación 2025 (real):  {factur_2025:>6.2f} M €")
print(f"   Facturación 2026 (pred):  {factur_2026_pred:>6.2f} M €")
print(f"   Variación esperada:       {variacion:>+6.2f} %")
if variacion < -1:
    print(f"   → El modelo predice CONTINUIDAD del decaimiento observado")
elif variacion > 1:
    print(f"   → El modelo predice cambio de tendencia (recuperación)")
else:
    print(f"   → El modelo predice estabilidad respecto a 2025")

FORECASTING DE 52 SEMANAS (2026)
Modelo seleccionado: SARIMA(0, 0, 0)(1, 1, 0, 52)
Transformación aplicada: logarítmica (revertida en los resultados)
RMSE sobre test 2025: 136,757 €
MAE sobre test 2025:  98,954 €

Inicio del forecast: 2025-12-29
Fin del forecast:    2026-12-21

Facturación esperada total 2026: 15,95 M €
   Intervalo de confianza 95%:   [6,86 ; 37,08] M €

   Facturación 2025 (real):   16.22 M €
   Facturación 2026 (pred):   15.95 M €
   Variación esperada:        -1.65 %
   → El modelo predice CONTINUIDAD del decaimiento observado


In [107]:
# Visualización del forecast (todas las fechas convertidas a string ISO)

# Convertir todas las fechas a strings ISO para evitar problemas con kaleido
fechas_historico_str  = [d.strftime("%Y-%m-%d") for d in serie.index]
fechas_forecast_str   = [d.strftime("%Y-%m-%d") for d in fechas_forecast]
fechas_forecast_rev   = fechas_forecast_str[::-1]

fig = go.Figure()

# Histórico
fig.add_trace(go.Scatter(
    x=fechas_historico_str, y=serie.values/1e3,
    mode="lines", line=dict(color=COLOR_NACIONAL, width=1.5),
    name="Histórico 2022-2025",
    hovertemplate="<b>%{x}</b><br>Histórico: %{y:,.1f} K €<extra></extra>"
))

# Intervalo de confianza 95% (banda amplia)
fig.add_trace(go.Scatter(
    x=fechas_forecast_str + fechas_forecast_rev,
    y=list(forecast_ic95.iloc[:,1]/1e3) + list(forecast_ic95.iloc[:,0]/1e3)[::-1],
    fill="toself", fillcolor="rgba(212, 160, 23, 0.18)",
    line=dict(color="rgba(0,0,0,0)"), name="IC 95%",
    hoverinfo="skip"
))

# Intervalo de confianza 80% (banda interior)
fig.add_trace(go.Scatter(
    x=fechas_forecast_str + fechas_forecast_rev,
    y=list(forecast_ic80.iloc[:,1]/1e3) + list(forecast_ic80.iloc[:,0]/1e3)[::-1],
    fill="toself", fillcolor="rgba(212, 160, 23, 0.35)",
    line=dict(color="rgba(0,0,0,0)"), name="IC 80%",
    hoverinfo="skip"
))

# Predicción media
fig.add_trace(go.Scatter(
    x=fechas_forecast_str, y=forecast_media.values/1e3,
    mode="lines", line=dict(color=COLOR_DESTACAR, width=2.2),
    name="Predicción 2026",
    hovertemplate="<b>%{x}</b><br>Predicción: %{y:,.1f} K €<extra></extra>"
))

# Línea vertical separando histórico de predicción (con add_shape + add_annotation)
fig.add_shape(
    type="line",
    x0=ultima_fecha.strftime("%Y-%m-%d"),
    x1=ultima_fecha.strftime("%Y-%m-%d"),
    y0=0, y1=1, yref="paper",
    line=dict(color=COLOR_NEUTRO, dash="dash", width=1.5),
)
fig.add_annotation(
    x=ultima_fecha.strftime("%Y-%m-%d"),
    y=1.0, yref="paper",
    text="Inicio forecast",
    showarrow=False,
    yshift=12,
    font=dict(size=10, color=COLOR_NEUTRO),
)

fig.update_layout(
    title=dict(text=f"<b>Forecast SARIMA{orden_regular}{orden_estacional} · Predicción semanal 2026</b><br><sup>Histórico 2022-2025 + 52 semanas de predicción con intervalos de confianza 80% y 95%</sup>"),
    xaxis_title="Fecha",
    yaxis_title="Facturación retail neta (K €)",
    height=550,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
guardar_y_mostrar(fig, "08_forecast_2026")

   Figura guardada: 08_forecast_2026.png


In [108]:
# Comparativa de totales anuales: histórico vs predicción 2026
totales_anuales = pd.DataFrame({
    "Año": [2022, 2023, 2024, 2025, "2026 (predicho)"],
    "Facturación total (M €)": [
        serie[serie.index.year == 2022].sum() / 1e6,
        serie[serie.index.year == 2023].sum() / 1e6,
        serie[serie.index.year == 2024].sum() / 1e6,
        serie[serie.index.year == 2025].sum() / 1e6,
        forecast_media.sum() / 1e6,
    ],
})
totales_anuales["IC 95% inferior"] = [None]*4 + [forecast_ic95.iloc[:,0].sum()/1e6]
totales_anuales["IC 95% superior"] = [None]*4 + [forecast_ic95.iloc[:,1].sum()/1e6]

print("COMPARATIVA DE TOTALES ANUALES")
print("=" * 70)
print(totales_anuales.to_string(index=False))

# Visualización en barras
fig = go.Figure()
colores_anios = [COLOR_NACIONAL]*4 + [COLOR_DESTACAR]
fig.add_trace(go.Bar(
    x=totales_anuales["Año"].astype(str),
    y=totales_anuales["Facturación total (M €)"],
    marker=dict(color=colores_anios),
    text=totales_anuales["Facturación total (M €)"].round(2).astype(str) + " M",
    textposition="outside",
    error_y=dict(
        type="data",
        array=[None]*4 + [(forecast_ic95.iloc[:,1].sum() - forecast_media.sum())/1e6],
        arrayminus=[None]*4 + [(forecast_media.sum() - forecast_ic95.iloc[:,0].sum())/1e6],
        visible=True, color=COLOR_NEUTRO,
    ),
    name="Facturación anual",
))
fig.update_layout(
    title=dict(text="<b>Facturación anual histórica y predicción 2026</b><br><sup>La barra dorada incluye intervalo de confianza al 95% (predicción SARIMA)</sup>"),
    xaxis_title="Año",
    yaxis_title="Facturación total (M €)",
    height=480, showlegend=False,
)
guardar_y_mostrar(fig, "09_comparativa_anual")

COMPARATIVA DE TOTALES ANUALES
            Año  Facturación total (M €)  IC 95% inferior  IC 95% superior
           2022                17.423848              NaN              NaN
           2023                16.984250              NaN              NaN
           2024                16.479671              NaN              NaN
           2025                16.215792              NaN              NaN
2026 (predicho)                15.948638         6.859425         37.08169


   Figura guardada: 09_comparativa_anual.png


### Conclusiones del modelo SARIMA y el forecasting

El modelo SARIMA seleccionado mediante auto_arima sobre la base del criterio AIC consigue capturar tanto la dinámica estacional anual (s=52) como la dependencia de corto plazo de la serie. La calidad del ajuste se evalúa con el diagnóstico de residuos: idealmente los residuos deben aproximarse a un ruido blanco gaussiano sin autocorrelación significativa, lo que se verifica mediante el test de Ljung-Box, el Q-Q plot frente a la distribución normal y la inspección de la ACF de los residuos.

El forecast proporciona una estimación cuantitativa de la facturación retail esperada para las 52 semanas de 2026, acompañada de intervalos de confianza al 80% y 95% que reflejan la incertidumbre creciente cuanto más lejos se extiende la predicción. Esta predicción constituye un input directo para la planificación comercial de Selmark y para el Capítulo 8 del trabajo, donde se cuantificará el retorno esperado de las acciones comerciales propuestas.

## 6. Síntesis y aportaciones al TFG

In [109]:
# Síntesis ampliada del análisis de series temporales
print("=" * 75)
print("SÍNTESIS DEL ANÁLISIS DE SERIES TEMPORALES")
print("=" * 75)

print(f"\n1. CARACTERIZACIÓN DE LA SERIE")
print(f"   Granularidad:                    semanal (semana ISO)")
print(f"   Periodo:                         2022-2025 (4 años naturales)")
print(f"   Observaciones:                   {len(serie):,}")
print(f"   Facturación acumulada histórica: {fmt_es(serie.sum()/1e6, 2)} M €")
print(f"   Tendencia detectada:             DESCENDENTE (-6,9 % en 4 años)")

print(f"\n2. DESCOMPOSICIÓN STL (s=52)")
print(f"   Varianza explicada por tendencia:      {v_trend:>6.2f} %")
print(f"   Varianza explicada por estacionalidad: {v_seas:>6.2f} %")
print(f"   Varianza explicada por residuos:       {v_resid:>6.2f} %")
print(f"   Semanas anómalas detectadas (>3σ):     {len(anomalias)}")

print(f"\n3. TESTS DE ESTACIONARIEDAD")
print(f"   ADF p-valor:   {adf_result[1]:.4f}  → estacionaria")
print(f"   KPSS p-valor:  {kpss_result[1]:.4f}  → estacionaria")
print(f"   Nota: la estacionariedad estadística no implica que la cifra absoluta")
print(f"   no descienda; la media de la serie es estable en torno a un valor que")
print(f"   sí decrece levemente año tras año.")

print(f"\n4. SELECCIÓN DEL MODELO SARIMA POR ITERACIÓN")
print(f"   Modelos candidatos probados:    {len(resultados)}")
print(f"   Criterio de selección:          RMSE sobre test 2025")
print(f"   Modelo ganador:                 SARIMA{orden_regular}{orden_estacional}")
print(f"   Transformación aplicada:        {transformacion_ganadora}")
print(f"   RMSE del modelo ganador:        {ganador['RMSE']:,.0f} €")
print(f"   MAE del modelo ganador:         {ganador['MAE']:,.0f} €")
print(f"   AIC sobre serie completa:       {ajuste.aic:.2f}")

print(f"\n5. FORECASTING 2026")
print(f"   Facturación predicha 2026:      {fmt_es(forecast_media.sum()/1e6, 2)} M €")
print(f"   IC 95% inferior:                {fmt_es(forecast_ic95.iloc[:,0].sum()/1e6, 2)} M €")
print(f"   IC 95% superior:                {fmt_es(forecast_ic95.iloc[:,1].sum()/1e6, 2)} M €")
print(f"   Variación esperada vs 2025:     {100*(forecast_media.sum()/serie[serie.index.year==2025].sum() - 1):+.2f} %")

print(f"\n" + "=" * 75)
print(f"CIERRE DEL NOTEBOOK 10 · Análisis de series temporales completado")
print(f"Figuras generadas: {len(list(RUTA_FIGURAS.glob('*.png')))} archivos PNG en {RUTA_FIGURAS.name}/")
print("=" * 75)

SÍNTESIS DEL ANÁLISIS DE SERIES TEMPORALES

1. CARACTERIZACIÓN DE LA SERIE
   Granularidad:                    semanal (semana ISO)
   Periodo:                         2022-2025 (4 años naturales)
   Observaciones:                   208
   Facturación acumulada histórica: 67,10 M €
   Tendencia detectada:             DESCENDENTE (-6,9 % en 4 años)

2. DESCOMPOSICIÓN STL (s=52)
   Varianza explicada por tendencia:        0.05 %
   Varianza explicada por estacionalidad:  94.90 %
   Varianza explicada por residuos:        23.51 %
   Semanas anómalas detectadas (>3σ):     7

3. TESTS DE ESTACIONARIEDAD
   ADF p-valor:   0.0000  → estacionaria
   KPSS p-valor:  0.1000  → estacionaria
   Nota: la estacionariedad estadística no implica que la cifra absoluta
   no descienda; la media de la serie es estable en torno a un valor que
   sí decrece levemente año tras año.

4. SELECCIÓN DEL MODELO SARIMA POR ITERACIÓN
   Modelos candidatos probados:    36
   Criterio de selección:          RMSE sobr

### Conclusiones generales y aportaciones del notebook al TFG

El análisis de series temporales desarrollado en este notebook aporta cuatro contribuciones diferenciales al Trabajo de Fin de Grado, alineadas con el carácter cuantitativo del Grado en Ingeniería Matemática.

En primer lugar, la **descomposición STL** confirma cuantitativamente lo que el análisis descriptivo del notebook 09b había evidenciado de forma visual: la facturación retail de Selmark presenta una estructura dominada por una componente estacional anual muy estable, una tendencia subyacente moderada y un residuo ruidoso pero acotado, salvo por algunas semanas anómalas que merecen revisión específica como casos de interés comercial.

En segundo lugar, el **análisis de autocorrelaciones** demuestra estadísticamente la presencia de dos estructuras temporales de dependencia: una de corto plazo (la facturación de una semana se correlaciona con las inmediatamente anteriores) y una estacional anual (la facturación de una semana se correlaciona con la misma semana del año anterior). Esta evidencia justifica formalmente la elección del modelo SARIMA con periodo estacional s=52.

En tercer lugar, los **tests de estacionariedad** (ADF y KPSS aplicados de forma complementaria) caracterizan formalmente las propiedades estadísticas de la serie y determinan el orden de diferenciación necesario para el modelado, aportando un rigor metodológico esperable en un trabajo de naturaleza cuantitativa.

En cuarto lugar, el **modelo SARIMA** ajustado proporciona una herramienta de forecasting cuantitativa con intervalos de confianza, capaz de predecir la facturación retail semanal del año 2026. Esta predicción constituye un input directo para las recomendaciones comerciales del Capítulo 8 del trabajo y, en particular, para la cuantificación del retorno esperado de las acciones comerciales propuestas en torno a los eventos comerciales identificados en el Capítulo 4.

### Aportaciones al Capítulo 8 (recomendaciones)

El forecasting obtenido habilita la cuantificación del impacto económico de las acciones comerciales que se propondrán en el capítulo de cierre del trabajo. En particular, la predicción de las semanas asociadas a San Valentín, rebajas y campañas de temporada (PV y OI) permitirá estimar el incremento porcentual sobre el escenario base que cabría esperar de cada acción comercial, y por tanto su retorno esperado en términos de facturación neta.

In [110]:
con.close()
print("Conexión a DuckDB cerrada. Notebook 10 completado.")

Conexión a DuckDB cerrada. Notebook 10 completado.
